In [2]:
# -*- coding: utf-8 -*-
"""
CÓDIGO MODIFICADO PARA CONSTRUIR BASE CON P558E2 y P558E3
SOLO SE MODIFICÓ: cols_500 (se agregaron "P558E2", "P558E3")
"""

import pandas as pd
import numpy as np
import os

# ============================================================
# CONFIGURACIÓN
# ============================================================
datos_originales = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO"
base_resultados = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25"
os.makedirs(base_resultados, exist_ok=True)

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def encontrar_archivo(carpeta, inicio_nombre):
    for f in os.listdir(carpeta):
        if f.upper().replace(".CSV", "").startswith(inicio_nombre.upper()):
            return os.path.join(carpeta, f)
    raise FileNotFoundError(f"No se encontró {inicio_nombre} en {carpeta}")

def leer_csv_inei(ruta):
    for sep in [",", ";"]:
        for enc in ["utf-8-sig", "latin1"]:
            try:
                df = pd.read_csv(ruta, sep=sep, encoding=enc, low_memory=False, dtype=str)
                if len(df.columns) > 1:
                    df.columns = df.columns.str.strip()
                    return df
            except Exception:
                pass
    raise ValueError(f"No se pudo leer el archivo: {ruta}")

def to_num(serie):
    return pd.to_numeric(serie, errors="coerce")

def construir_llave_persona(df, llaves=None, nombre="llave_persona"):
    if llaves is None:
        llaves = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
    df = df.copy()
    for c in llaves:
        df[f"_{c}_k"] = pd.to_numeric(df[c], errors="coerce").astype("Int64").astype(str)
    df[nombre] = df[[f"_{c}_k" for c in llaves]].agg("-".join, axis=1)
    return df.drop(columns=[f"_{c}_k" for c in llaves])

# ============================================================
# COLUMNAS PARA USO DE BILLETERA
# ============================================================
COLS_USO_BILLETERA = [f"P558H{i}_7" for i in range(1, 13)]

def reparar_variables(df):
    df = df.copy()

    df["Edad"] = to_num(df["P208A"]) if "P208A" in df.columns else df.get("Edad")
    df["P507_num"] = to_num(df["P507"])
    df["P510A1_num"] = to_num(df["P510A1"])
    df["P511A_num"] = to_num(df["P511A"])

    # --- Ocupado ---
    df["Ocupado"] = df["P507_num"].notna().astype(int)

    # --- Informalidad ---
    tfnr = df["P507_num"] == 5
    trab_hogar = df["P507_num"] == 6
    sin_sunat = (df["P507_num"] == 2) & (df["P510A1_num"] == 3)
    dependiente = df["P507_num"].isin([3, 4])
    sin_contrato = df["P511A_num"] == 7

    df["Informal"] = (
        sin_sunat.fillna(False)
        | tfnr.fillna(False)
        | trab_hogar.fillna(False)
        | (dependiente & sin_contrato.fillna(False))
    ).astype(int)
    df.loc[df["Ocupado"] == 0, "Informal"] = pd.NA

    # --- Crédito formal ---
    df["P558E1_4"] = to_num(df.get("P558E1_4")).fillna(0)
    df["P558E1_9"] = to_num(df.get("P558E1_9")).fillna(0)
    df["CreditoFormal"] = ((df["P558E1_4"] == 4) | (df["P558E1_9"] == 9)).astype(int)

    # --- Tenencia de billetera ---
    df["P558E1_8"] = to_num(df.get("P558E1_8")).fillna(0)
    df["TenenciaBilletera"] = (df["P558E1_8"] == 8).astype(int)

    # --- USO DE BILLETERA ---
    cols_uso_presentes = [c for c in COLS_USO_BILLETERA if c in df.columns]
    if cols_uso_presentes:
        uso_flags = pd.concat(
            [to_num(df[c]).fillna(0) == 7 for c in cols_uso_presentes], axis=1
        )
        df["UsoBilletera"] = uso_flags.any(axis=1).astype(int)
    else:
        df["UsoBilletera"] = pd.NA

    # --- Crédito informal ---
    if "P558G7" in df.columns:
        df["CreditoInformal"] = (to_num(df["P558G7"]).fillna(0) == 7).astype(int)

    # =========================================================
    # NUEVO: P558E2 y P558E3 (Solicitud y denegación de crédito)
    # =========================================================
    if "P558E2" in df.columns:
        df["P558E2_num"] = to_num(df["P558E2"])
    if "P558E3" in df.columns:
        df["P558E3_num"] = to_num(df["P558E3"])

    return df

def construir_master(anio, carpetas):
    print(f"Cargando módulos {anio}...")

    ruta_200 = encontrar_archivo(*carpetas["Mod200"])
    df200 = leer_csv_inei(ruta_200)[["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "P203", "P207", "P208A"]]

    ruta_300 = encontrar_archivo(*carpetas["Mod300"])
    df300 = leer_csv_inei(ruta_300)[["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "P301A", "P301B"]]

    ruta_500 = encontrar_archivo(*carpetas["Mod500"])
    df500_full = leer_csv_inei(ruta_500)
    # =========================================================
    # MODIFICACIÓN CLAVE: se agregaron "P558E2" y "P558E3"
    # =========================================================
    cols_500 = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO",
                "P507", "P510A1", "P511A", "P512A",
                "P558E1_4", "P558E1_8", "P558E1_9",
                "P558E2", "P558E3",  # <--- NUEVO
                "P558G7"] + COLS_USO_BILLETERA
    df500 = df500_full[[c for c in cols_500 if c in df500_full.columns]]

    ruta_100 = encontrar_archivo(*carpetas["Mod100"])
    df100 = leer_csv_inei(ruta_100)
    cols_100 = ["CONGLOME", "VIVIENDA", "HOGAR", "DOMINIO", "ESTRATO", "PANEL"]
    for fac in ["FACTOR", "FACPOB", "FACTOR07"]:
        if fac in df100.columns:
            cols_100.append(fac)
            break
    df100 = df100[cols_100]

    ruta_sum = encontrar_archivo(*carpetas["Sumaria"])
    df_sum = leer_csv_inei(ruta_sum)
    cols_sum = ["CONGLOME", "VIVIENDA", "HOGAR"]
    gasto_encontrada = None
    for g in ["GASHOG2D", "GASHOG1D", "GASHOG2", "GASHOG1"]:
        if g in df_sum.columns:
            gasto_encontrada = g
            break
    if gasto_encontrada:
        df_sum = df_sum.rename(columns={gasto_encontrada: "GASHOG2"})
        cols_sum.append("GASHOG2")
    for m in ["MIEPERHO", "MIEMBRO"]:
        if m in df_sum.columns:
            cols_sum.append(m)
            break
    df_sum = df_sum[cols_sum]

    llave_persona = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
    llave_hogar = ["CONGLOME", "VIVIENDA", "HOGAR"]

    df = df200.merge(df300, on=llave_persona, how="left")
    df = df.merge(df500, on=llave_persona, how="left")
    df = df.merge(df100, on=llave_hogar, how="left")
    df = df.merge(df_sum, on=llave_hogar, how="left")

    df = construir_llave_persona(df)
    df = reparar_variables(df)

    ruta_salida = os.path.join(base_resultados, f"MASTER_{anio}.csv")
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"✅ Guardado: {ruta_salida}  ({df.shape[0]:,} x {df.shape[1]})")
    return df

# ============================================================
# 1. CONSTRUIR MASTER 2024 y 2025
# ============================================================

carpetas_2024 = {
    "Mod100": (os.path.join(datos_originales, "966-Modulo01"), "Enaho01-2024-100"),
    "Mod200": (os.path.join(datos_originales, "966-Modulo02"), "Enaho01-2024-200"),
    "Mod300": (os.path.join(datos_originales, "966-Modulo03"), "Enaho01A-2024-300"),
    "Mod500": (os.path.join(datos_originales, "966-Modulo05"), "Enaho01a-2024-500"),
    "Sumaria": (os.path.join(datos_originales, "966-Modulo34"), "Sumaria-2024-12g"),
}
df_master_2024 = construir_master("2024", carpetas_2024)

carpetas_2025 = {
    "Mod100": (os.path.join(datos_originales, "1031-Modulo01-2025"), "Enaho01-2025-100"),
    "Mod200": (os.path.join(datos_originales, "1031-Modulo02-2025"), "Enaho01-2025-200"),
    "Mod300": (os.path.join(datos_originales, "1031-Modulo03-2025"), "Enaho01A-2025-300"),
    "Mod500": (os.path.join(datos_originales, "1031-Modulo05-2025"), "Enaho01a-2025-500"),
    "Sumaria": (os.path.join(datos_originales, "1031-Modulo34-2025"), "Sumaria-2025-12g"),
}
df_master_2025 = construir_master("2025", carpetas_2025)

# ============================================================
# 2. CONSTRUIR PANEL
# ============================================================

df24 = pd.read_csv(os.path.join(base_resultados, "MASTER_2024.csv"), encoding="utf-8-sig", low_memory=False)
df25 = pd.read_csv(os.path.join(base_resultados, "MASTER_2025.csv"), encoding="utf-8-sig", low_memory=False)

df24 = construir_llave_persona(df24)
df25 = construir_llave_persona(df25)

match = df24[["llave_persona"]].drop_duplicates().merge(
    df25[["llave_persona"]].drop_duplicates(), on="llave_persona", how="inner"
)

cols_2024 = ["llave_persona", "P203", "P207", "Edad", "Ocupado", "Informal",
             "TenenciaBilletera", "UsoBilletera", "CreditoFormal", "FACTOR07", "P301A",
             "ESTRATO", "DOMINIO", "MIEPERHO", "CONGLOME",
             "P558E2_num", "P558E3_num"]  # <--- NUEVO
df_2024_sub = df24[df24["llave_persona"].isin(match["llave_persona"])][cols_2024].rename(
    columns={"CreditoFormal": "CreditoFormal_2024"}
)

cols_2025 = ["llave_persona", "CreditoFormal"]
if "CreditoInformal" in df25.columns:
    cols_2025.append("CreditoInformal")
df_2025_sub = df25[cols_2025].rename(
    columns={"CreditoFormal": "CreditoFormal_2025", "CreditoInformal": "CreditoInformal_2025"}
)

panel_df = df_2024_sub.merge(df_2025_sub, on="llave_persona", how="left")
panel_df.to_csv(os.path.join(base_resultados, "PANEL_2024_2025.csv"), index=False, encoding="utf-8-sig")
print(f"✅ PANEL_2024_2025.csv guardado ({panel_df.shape[0]:,} filas)")

# ============================================================
# 3. FILTRAR MUESTRA ANALÍTICA
# ============================================================

df = pd.read_csv(os.path.join(base_resultados, "PANEL_2024_2025.csv"), encoding="utf-8-sig", low_memory=False)

df["P203_num"] = pd.to_numeric(df["P203"], errors="coerce")
df["Ocupado_num"] = pd.to_numeric(df["Ocupado"], errors="coerce")
df["CreditoFormal_2024_num"] = pd.to_numeric(df["CreditoFormal_2024"], errors="coerce")
df["CreditoFormal_2025_num"] = pd.to_numeric(df["CreditoFormal_2025"], errors="coerce")
df["Informal_num"] = pd.to_numeric(df["Informal"], errors="coerce")
df["P301A_num"] = pd.to_numeric(df["P301A"], errors="coerce")

# Excluir códigos 99 en nivel educativo
df = df[df["P301A_num"] != 99]

df = df[df["P203_num"] == 1]
df = df[df["Ocupado_num"] == 1]
df = df[df["CreditoFormal_2024_num"] == 0]
df = df.dropna(subset=["Informal_num", "TenenciaBilletera", "UsoBilletera", "P207", "Edad", "P301A"])

df["NuevoCredito"] = (df["CreditoFormal_2025_num"] == 1).astype(int)

# ============================================================
# 4. RENOMBRAR VARIABLES FINALES
# ============================================================

diccionario_renombres = {
    "llave_persona": "id_persona",
    "P203": "jefe_hogar",
    "P207": "sexo",
    "Edad": "edad",
    "P301A": "nivel_educativo",
    "Ocupado": "ocupado",
    "Informal": "trabajador_informal",
    "TenenciaBilletera": "tiene_billetera",
    "UsoBilletera": "usa_billetera",
    "CreditoFormal_2024": "credito_formal_2024",
    "CreditoFormal_2025": "credito_formal_2025",
    "CreditoInformal_2025": "credito_informal_2025",
    "NuevoCredito": "nuevo_credito_formal",
    "FACTOR07": "factor_expansion",
    "ESTRATO": "estrato",
    "DOMINIO": "dominio",
    "MIEPERHO": "miembros_hogar",
    "CONGLOME": "conglomerado",
    "P558E2_num": "solicito_credito",
    "P558E3_num": "obtuvo_credito",
}

columnas_a_renombrar = {k: v for k, v in diccionario_renombres.items() if k in df.columns}
df = df.rename(columns=columnas_a_renombrar)

columnas_auxiliares = ["P203_num", "Ocupado_num", "CreditoFormal_2024_num", 
                       "CreditoFormal_2025_num", "Informal_num", "P301A_num"]
columnas_a_eliminar = [col for col in columnas_auxiliares if col in df.columns]
df = df.drop(columns=columnas_a_eliminar)

# ============================================================
# 5. GUARDAR BASE FINAL
# ============================================================

df.to_csv(os.path.join(base_resultados, "BASE_REGRESIONES.csv"), index=False, encoding="utf-8-sig")
print(f"✅ BASE_REGRESIONES.csv guardado ({df.shape[0]:,} filas)")

print("\n" + "="*70)
print("VARIABLES FINALES EN BASE_REGRESIONES.csv")
print("="*70)
for i, col in enumerate(df.columns, 1):
    print(f"{i:>3}. {col}")

print("\n" + "="*70)
print("VERIFICACIÓN RÁPIDA DE VARIABLES NUEVAS")
print("="*70)

if 'solicito_credito' in df.columns:
    print(f"🔍 solicito_credito: {df['solicito_credito'].sum():,} personas ({df['solicito_credito'].mean():.2%})")
else:
    print("⚠️ solicito_credito no encontrada")

if 'obtuvo_credito' in df.columns:
    print(f"🔍 obtuvo_credito: {df['obtuvo_credito'].sum():,} personas ({df['obtuvo_credito'].mean():.2%})")
else:
    print("⚠️ obtuvo_credito no encontrada")

Cargando módulos 2024...
✅ Guardado: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\MASTER_2024.csv  (117,721 x 46)
Cargando módulos 2025...
✅ Guardado: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\MASTER_2025.csv  (115,145 x 46)


KeyError: "['P558E2_num', 'P558E3_num'] not in index"

In [3]:
# -*- coding: utf-8 -*-
"""
EXPLORAR ARCHIVOS ORIGINALES ENAHO - BUSCAR P558E2 y P558E3
"""

import pandas as pd
import os

# ============================================================
# CONFIGURACIÓN
# ============================================================
datos_originales = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO"

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def encontrar_archivo(carpeta, inicio_nombre):
    for f in os.listdir(carpeta):
        if f.upper().replace(".CSV", "").startswith(inicio_nombre.upper()):
            return os.path.join(carpeta, f)
    return None

def leer_columnas(ruta):
    """Lee solo los nombres de las columnas de un CSV, sin cargar todo el archivo"""
    try:
        for sep in [",", ";"]:
            for enc in ["utf-8-sig", "latin1"]:
                try:
                    df_sample = pd.read_csv(ruta, sep=sep, encoding=enc, nrows=0)
                    return df_sample.columns.tolist()
                except:
                    pass
    except:
        pass
    return None

# ============================================================
# 1. BUSCAR ARCHIVOS MÓDULO 500 (donde deberían estar P558)
# ============================================================

print("="*70)
print("BUSCANDO ARCHIVOS MÓDULO 500 (ENAHO 2024 y 2025)")
print("="*70)

# Estructura de carpetas
carpeta_2024 = os.path.join(datos_originales, "966-Modulo05")
carpeta_2025 = os.path.join(datos_originales, "1031-Modulo05-2025")

print(f"\n📁 Carpeta 2024: {carpeta_2024}")
print(f"📁 Carpeta 2025: {carpeta_2025}")

# Buscar archivos en cada carpeta
for anio, carpeta in [("2024", carpeta_2024), ("2025", carpeta_2025)]:
    print(f"\n{'='*50}")
    print(f"🔍 BUSCANDO EN {anio.upper()}: {carpeta}")
    print('='*50)
    
    if not os.path.exists(carpeta):
        print(f"❌ Carpeta no existe: {carpeta}")
        continue
    
    archivos = os.listdir(carpeta)
    print(f"\n📄 Archivos encontrados: {len(archivos)}")
    for f in archivos:
        print(f"  • {f}")
    
    # Buscar archivos que contengan "500" o "Enaho"
    archivos_500 = [f for f in archivos if '500' in f or 'Enaho' in f]
    
    for archivo in archivos_500:
        ruta_completa = os.path.join(carpeta, archivo)
        print(f"\n📄 Revisando: {archivo}")
        
        columnas = leer_columnas(ruta_completa)
        if columnas is None:
            print(f"  ❌ No se pudo leer el archivo")
            continue
        
        print(f"  ✅ {len(columnas)} columnas encontradas")
        
        # Buscar P558 específicamente
        p558_cols = [c for c in columnas if 'P558' in c]
        if p558_cols:
            print(f"\n  🔍 Columnas P558 encontradas ({len(p558_cols)}):")
            for c in sorted(p558_cols):
                print(f"    • {c}")
            
            # Buscar específicamente P558E2 y P558E3
            if 'P558E2' in columnas:
                print("    ✅ ¡P558E2 ENCONTRADA!")
            else:
                print("    ❌ P558E2 NO encontrada")
            
            if 'P558E3' in columnas:
                print("    ✅ ¡P558E3 ENCONTRADA!")
            else:
                print("    ❌ P558E3 NO encontrada")
        else:
            print("  ⚠️ No se encontraron columnas P558 en este archivo")

# ============================================================
# 2. RESUMEN FINAL
# ============================================================

print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)

print("""
Si P558E2 y P558E3 NO aparecen en la lista de columnas de ningún archivo,
significa que estas variables NO existen en tu versión de ENAHO.

En ese caso, la única solución es:
1. ELIMINAR la afirmación incorrecta del documento
2. Corregir la sección de limitaciones

Si APARECEN en algún archivo, entonces el problema es que tu script
de construcción no las está extrayendo correctamente.
""")

BUSCANDO ARCHIVOS MÓDULO 500 (ENAHO 2024 y 2025)

📁 Carpeta 2024: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO\966-Modulo05
📁 Carpeta 2025: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO\1031-Modulo05-2025

🔍 BUSCANDO EN 2024: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO\966-Modulo05

📄 Archivos encontrados: 5
  • CED-01A-500 ENAHO 2024.pdf
  • DefinicionValoresMonetariosBasedeDatos.pdf
  • Diccionario2024.pdf
  • Enaho01a-2024-500.csv
  • FichaTecnica.pdf

📄 Revisando: CED-01A-500 ENAHO 2024.pdf
  ✅ 1 columnas encontradas
  ⚠️ No se encontraron columnas P558 en este archivo

📄 Revisando: Enaho01a-2024-500.csv
  ✅ 1425 columnas encontradas

  🔍 Columnas P558 encontradas (135):
    • P55810A
    • P55810B
    • P5581A
    • P5581B
    • P5582A
    • P5582B
    • P5583A
    • P5583B
    • P5584A
    • P5584B
    • P5585A
    • P5585B
    • P5586A
    • P5586B
    • P5587A
    • P5587B
    • P5588A


In [4]:
# -*- coding: utf-8 -*-
"""
VERIFICAR VARIABLES DE GASTO EN MASTER_2024 y MASTER_2025
"""

import pandas as pd
import os

# ============================================================
# CONFIGURACIÓN
# ============================================================
base_resultados = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25"

# ============================================================
# 1. VERIFICAR EN MASTER_2024
# ============================================================

print("="*70)
print("VERIFICANDO MASTER_2024.csv")
print("="*70)

ruta_2024 = os.path.join(base_resultados, "MASTER_2024.csv")
if os.path.exists(ruta_2024):
    df24 = pd.read_csv(ruta_2024, encoding="utf-8-sig", nrows=5)
    print(f"✅ MASTER_2024.csv encontrado")
    print(f"\n📊 Columnas relacionadas con gasto:")
    
    # Buscar columnas con GAS, MIEP, MIEM
    cols_gasto = [c for c in df24.columns if 'GAS' in c or 'MIEP' in c or 'MIEM' in c]
    for col in cols_gasto:
        print(f"  • {col}")
    
    # Verificar específicamente las variables que buscamos
    print(f"\n📊 Variables específicas:")
    for var in ['GASHOG1D', 'GASHOG2D', 'GASHOG1', 'GASHOG2', 'MIEPERHO', 'MIEMBRO']:
        if var in df24.columns:
            print(f"  ✅ {var} ENCONTRADA")
        else:
            print(f"  ❌ {var} NO encontrada")
else:
    print("❌ MASTER_2024.csv no encontrado")

# ============================================================
# 2. VERIFICAR EN MASTER_2025
# ============================================================

print("\n" + "="*70)
print("VERIFICANDO MASTER_2025.csv")
print("="*70)

ruta_2025 = os.path.join(base_resultados, "MASTER_2025.csv")
if os.path.exists(ruta_2025):
    df25 = pd.read_csv(ruta_2025, encoding="utf-8-sig", nrows=5)
    print(f"✅ MASTER_2025.csv encontrado")
    print(f"\n📊 Columnas relacionadas con gasto:")
    
    # Buscar columnas con GAS, MIEP, MIEM
    cols_gasto = [c for c in df25.columns if 'GAS' in c or 'MIEP' in c or 'MIEM' in c]
    for col in cols_gasto:
        print(f"  • {col}")
    
    # Verificar específicamente las variables que buscamos
    print(f"\n📊 Variables específicas:")
    for var in ['GASHOG1D', 'GASHOG2D', 'GASHOG1', 'GASHOG2', 'MIEPERHO', 'MIEMBRO']:
        if var in df25.columns:
            print(f"  ✅ {var} ENCONTRADA")
        else:
            print(f"  ❌ {var} NO encontrada")
else:
    print("❌ MASTER_2025.csv no encontrado")

# ============================================================
# 3. VERIFICAR EN BASE_REGRESIONES
# ============================================================

print("\n" + "="*70)
print("VERIFICANDO BASE_REGRESIONES.csv")
print("="*70)

ruta_base = os.path.join(base_resultados, "BASE_REGRESIONES.csv")
if os.path.exists(ruta_base):
    df_base = pd.read_csv(ruta_base, encoding="utf-8-sig", nrows=5)
    print(f"✅ BASE_REGRESIONES.csv encontrado")
    
    print(f"\n📊 Columnas relacionadas con gasto:")
    cols_gasto = [c for c in df_base.columns if 'GAS' in c or 'MIEP' in c or 'MIEM' in c]
    if cols_gasto:
        for col in cols_gasto:
            print(f"  • {col}")
    else:
        print("  ⚠️ No hay columnas de gasto en BASE_REGRESIONES")
        print("  → Esto es esperable, porque solo exportaste variables específicas")
else:
    print("❌ BASE_REGRESIONES.csv no encontrado")

# ============================================================
# 4. RESUMEN
# ============================================================

print("\n" + "="*70)
print("RESUMEN - ¿QUÉ VARIABLES TENEMOS?")
print("="*70)

print("""
📌 Si GASHOG2D y MIEPERHO están en MASTER_2024 y MASTER_2025:
   → Podemos crear log_gasto_percapita en el script de construcción
   → Hay que modificar el código para que las incluya en BASE_REGRESIONES

📌 Si NO están en MASTER_2024 y MASTER_2025:
   → Significa que tu script no las está extrayendo del módulo Sumaria
   → Hay que corregir el script de construcción para incluirlas

📌 Si están en MASTER pero NO en BASE_REGRESIONES:
   → Hay que agregarlas al panel y luego a la base final
""")

VERIFICANDO MASTER_2024.csv
✅ MASTER_2024.csv encontrado

📊 Columnas relacionadas con gasto:
  • GASHOG2
  • MIEPERHO

📊 Variables específicas:
  ❌ GASHOG1D NO encontrada
  ❌ GASHOG2D NO encontrada
  ❌ GASHOG1 NO encontrada
  ✅ GASHOG2 ENCONTRADA
  ✅ MIEPERHO ENCONTRADA
  ❌ MIEMBRO NO encontrada

VERIFICANDO MASTER_2025.csv
✅ MASTER_2025.csv encontrado

📊 Columnas relacionadas con gasto:
  • GASHOG2
  • MIEPERHO

📊 Variables específicas:
  ❌ GASHOG1D NO encontrada
  ❌ GASHOG2D NO encontrada
  ❌ GASHOG1 NO encontrada
  ✅ GASHOG2 ENCONTRADA
  ✅ MIEPERHO ENCONTRADA
  ❌ MIEMBRO NO encontrada

VERIFICANDO BASE_REGRESIONES.csv
✅ BASE_REGRESIONES.csv encontrado

📊 Columnas relacionadas con gasto:
  ⚠️ No hay columnas de gasto en BASE_REGRESIONES
  → Esto es esperable, porque solo exportaste variables específicas

RESUMEN - ¿QUÉ VARIABLES TENEMOS?

📌 Si GASHOG2D y MIEPERHO están en MASTER_2024 y MASTER_2025:
   → Podemos crear log_gasto_percapita en el script de construcción
   → Hay que modif